# VSPAERO mesh equalization

この Notebook は、**基準 Wing の chordwise mesh size を基準に、各 surface Geom の物理的な U/W mesh spacing を揃えた `.vsp3` を作る**ための実行例です。

通常変更するのは **「2. Settings」だけ**です。VSPAERO 解析、mesh convergence study、post-intersection mesh quality 診断はこの Notebook では行いません。


## 1. Setup

リポジトリ内の `src/VSPAEROMesh.py` を読み込みます。通常、このセルは変更しません。


In [ ]:
from pathlib import Path
import sys

from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "VSPAEROMesh.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/VSPAEROMesh.py. "
        "Run this notebook inside the repository."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.VSPAEROMesh import equalize_vspaero_tessellation

print("repo_root:", repo_root)


## 2. Settings

**通常変更するのはこのセルだけです。**

- `model_name`: `examples/models/<model_name>/<model_name>.vsp3` を入力モデルとして使います。
- `reference_wing_name`: mesh size の基準にする Wing の Geom 名です。
- `reference_tess_w`: reference Wing に与える `Tess_W` です。その実3次元 W-edge median が target mesh size になります。
- `geom_names=None`: 調整可能な surface Geom をすべて対象にします。対象を限定したい場合だけ Geom 名のリストを指定します。
- `tolerance` / `max_iterations`: 通常は既定値のままで構いません。


In [ ]:
model_name = "Boeing_777-9x_mod"
reference_wing_name = "Wing"
reference_tess_w = 33

geom_names = None
tolerance = 0.10
max_iterations = 4

input_vsp3_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)

output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "mesh_equalization"
    / "results"
    / model_name
)
output_vsp3_path = output_dir / f"{model_name}.equalized.vsp3"
report_path = output_dir / "equalization_report.csv"

print("input_vsp3_path:", input_vsp3_path)
print("output_vsp3_path:", output_vsp3_path)


## 3. Run equalization

`equalize_vspaero_tessellation()` が `.vsp3` を読み、物理的な edge size を測定し、`Tess_W`、`SectTess_U` / `Tess_U`、active Wing end cap の `CapUMinTess` を調整します。clustering parameter は変更しません。


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)

report = equalize_vspaero_tessellation(
    input_vsp3_path=input_vsp3_path,
    output_vsp3_path=output_vsp3_path,
    reference_wing_name=reference_wing_name,
    reference_tess_w=reference_tess_w,
    geom_names=geom_names,
    tolerance=tolerance,
    max_iterations=max_iterations,
)

report.to_csv(report_path, index=False)


## 4. Results

まず `u_target_ratio` と `w_target_ratio` を確認します。1.0 に近いほど、reference Wing から得た target mesh size に近い物理 edge size です。Wing end cap が有効な場合は `cap_target_ratio` も確認します。

詳細が必要な場合は、下の `report` 全体に `before_*` / `after_*` の値が入っています。


In [ ]:
summary_columns = [
    "geom_name",
    "geom_type",
    "status",
    "after_tess_w",
    "after_u_tess",
    "after_cap_tess",
    "target_edge_size",
    "u_target_ratio",
    "w_target_ratio",
    "cap_target_ratio",
]

display(report[[column for column in summary_columns if column in report.columns]])


## 5. Output

生成物は次の2つです。

- equalized `.vsp3`
- before / after と実測 edge size を含む `equalization_report.csv`

この Notebook の役割はここまでです。equalized model の VSPAERO convergence や結果妥当性の評価は、別の解析として行ってください。


In [ ]:
print("equalized model:", output_vsp3_path)
print("report:", report_path)
